# Figure 3

This notebook generates, processes, and visualises all subfigures for Figure 3 in one place. It writes Figure-3-specific intermediate pickle payloads into `raw_data/` and `processed_data/`, then saves the composed figure into `figures/pdf/`, `figures/png/`, and `figures/eps/`.


## Setup

Load the simulation, processing, and plotting tools used by the standalone Figure 3 workflow.


In [ ]:
%load_ext autoreload
%autoreload 2

import pickle
from pathlib import Path

import jax.random as jr
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.gridspec import GridSpec
from tqdm.auto import tqdm

from slide.data_generation import nk_grid_pairs, run_nk_diffusion_replicates
from slide.data_processing import (
    nk_metric_comparison_from_accuracy,
    process_mutation_accuracy,
    process_popsize_accuracy,
    process_ruggedness_accuracy,
    smooth_rugged_example,
)
from slide.utils import get_figures_dir, get_processed_data_dir, get_raw_data_dir, load_pickle, save_pickle

RAW_DATA_DIR = get_raw_data_dir()
PROCESSED_DATA_DIR = get_processed_data_dir()
FIGURES_DIR = get_figures_dir()
for figure_type in ("pdf", "png", "eps"):
    (FIGURES_DIR / figure_type).mkdir(parents=True, exist_ok=True)

print(f"raw_data: {RAW_DATA_DIR}")
print(f"processed_data: {PROCESSED_DATA_DIR}")
print(f"figures: {FIGURES_DIR}")


## Figure 3 Files And Parameters

Define all filenames and run-defining parameters used by the raw generation and processing sections. The raw and processed filenames are Figure-3-specific so this notebook does not overwrite outputs from the three-notebook pipeline.


In [ ]:
RAW_FILES = {
    "nk_decay_grid": "figure3_nk_decay_grid_raw.pkl",
    "nk_popsize_accuracy": "figure3_nk_popsize_accuracy_raw.pkl",
    "nk_mutation_accuracy": "figure3_nk_mutation_accuracy_raw.pkl",
}

PROCESSED_FILES = {
    "decay_examples": "figure3_decay_examples_processed.pkl",
    "ruggedness_accuracy": "figure3_ruggedness_accuracy_processed.pkl",
    "popsize_accuracy": "figure3_popsize_accuracy_processed.pkl",
    "mutation_accuracy": "figure3_mutation_accuracy_processed.pkl",
    "metric_comparison": "figure3_metric_comparison_processed.pkl",
}

# NK grid decay parameters for panels A, B, and E.
grid_params = {
    "N_range": (10, 50),
    "num_grid_samples": 10,
    "A": 2,
    "mutation_rate": 0.5,
    "popsize": 2500,
    "num_landscapes": 25,
    "num_reps_per_landscape": 10,
    "M": 25,
    "seed": 42,
    "start_policy": "zero",
}
grid_params["nk_pairs"] = nk_grid_pairs(grid_params["N_range"], grid_params["num_grid_samples"])

# NK accuracy sweep parameters for panels C and D.
accuracy_params = {
    "N": 25,
    "K": 15,
    "A": 2,
    "num_lscapes": 25,
    "num_reps_per_lscape": 20,
    "M": 25,
    "seed": 42,
    "random_starting_genotype": True,
    "start_policy": "random",
    "fit_mutation_scale": 1.0,
}
accuracy_params["pop_sizes"] = np.logspace(start=1, stop=6, num=6, endpoint=True, dtype=int)
accuracy_params["popsize_mutation_rate"] = 0.5
accuracy_params["mutation_sweep_popsize"] = 2000
accuracy_params["mutation_rates"] = np.linspace(0.01, 2, 25)

rho_nk_accuracy_true = (accuracy_params["K"] + 1) / accuracy_params["N"]


## Raw Data Generation

Generate the raw NK diffusion products needed for all Figure 3 panels and save them as payload dictionaries with `data`, `params`, and `metadata`.


### NK Decay Grid - Panels A, B, And E

Generate the large NK no-selection diffusion grid used for example decay curves, ruggedness prediction accuracy, and metric comparison. Each diffusion starts from the all-zero genotype and records mean population fitness trajectories $F_\mu$ over generations $M$.


In [ ]:
rep_keys = jr.split(jr.PRNGKey(grid_params["seed"]), grid_params["num_landscapes"])
nk_decay_grid = []

for N, K in tqdm(grid_params["nk_pairs"], desc="Figure 3 NK decay grid"):
    pair_results = []
    for key in rep_keys:
        run = run_nk_diffusion_replicates(
            key,
            n_sites=N,
            k=K,
            num_alleles=grid_params["A"],
            start=np.zeros(N, dtype=np.int32),
            popsize=grid_params["popsize"],
            mutation_rate=grid_params["mutation_rate"] / N,
            num_reps=grid_params["num_reps_per_landscape"],
            num_steps=grid_params["M"],
        )
        pair_results.append(run["fitness"].mean(axis=-1))
    nk_decay_grid.append(np.asarray(pair_results))

nk_decay_grid_payload = {
    "data": np.asarray(nk_decay_grid),
    "params": grid_params,
    "metadata": {
        "description": "Raw NK no-selection diffusion grid for Figure 3 panels A, B, and E.",
        "paper_reference": "Figure 3A-B/E",
        "output_key": "figure3_nk_decay_grid",
        "filename": RAW_FILES["nk_decay_grid"],
    },
}
save_pickle(nk_decay_grid_payload, RAW_DATA_DIR / RAW_FILES["nk_decay_grid"])


### Population-Size Accuracy Sweep - Panel C

Generate the population-size sensitivity sweep for fitted ruggedness $\rho_{fit}$. Each run uses random starting genotypes, matching the current edited Figure 3 generation parameters.


In [ ]:
rep_keys = jr.split(jr.PRNGKey(accuracy_params["seed"]), accuracy_params["num_lscapes"])
popsize_accuracy = []

for popsize in tqdm(accuracy_params["pop_sizes"], desc="Figure 3 popsize accuracy"):
    pop_results = []
    for key in rep_keys:
        start = (
            np.asarray(jr.randint(key, (accuracy_params["N"],), 0, accuracy_params["A"]), dtype=np.int32)
            if accuracy_params["random_starting_genotype"]
            else np.zeros(accuracy_params["N"], dtype=np.int32)
        )
        run = run_nk_diffusion_replicates(
            key,
            n_sites=accuracy_params["N"],
            k=accuracy_params["K"],
            num_alleles=accuracy_params["A"],
            start=start,
            popsize=int(popsize),
            mutation_rate=accuracy_params["popsize_mutation_rate"] / accuracy_params["N"],
            num_reps=accuracy_params["num_reps_per_lscape"],
            num_steps=accuracy_params["M"],
        )
        pop_results.append(run["fitness"].mean(axis=-1))
    popsize_accuracy.append(np.asarray(pop_results))

popsize_params = {
    "N": accuracy_params["N"],
    "K": accuracy_params["K"],
    "A": accuracy_params["A"],
    "num_lscapes": accuracy_params["num_lscapes"],
    "num_reps_per_lscape": accuracy_params["num_reps_per_lscape"],
    "M": accuracy_params["M"],
    "seed": accuracy_params["seed"],
    "start_policy": accuracy_params["start_policy"],
    "random_starting_genotype": accuracy_params["random_starting_genotype"],
    "pop_sizes": accuracy_params["pop_sizes"],
    "mutation_rate": accuracy_params["popsize_mutation_rate"],
    "fit_mutation_scale": accuracy_params["fit_mutation_scale"],
    "rho_NK": rho_nk_accuracy_true,
}
popsize_accuracy_payload = {
    "data": np.asarray(popsize_accuracy),
    "params": popsize_params,
    "metadata": {
        "description": "Raw NK population-size sensitivity sweep for Figure 3C.",
        "paper_reference": "Figure 3C",
        "output_key": "figure3_nk_popsize_accuracy",
        "filename": RAW_FILES["nk_popsize_accuracy"],
    },
}
save_pickle(popsize_accuracy_payload, RAW_DATA_DIR / RAW_FILES["nk_popsize_accuracy"])


### Mutation-Rate Accuracy Sweep - Panel D

Generate the mutation-rate sensitivity sweep for fitted ruggedness $\rho_{fit}$. The x-axis values are total mutations per cell per generation, $\theta$.


In [ ]:
rep_keys = jr.split(jr.PRNGKey(accuracy_params["seed"]), accuracy_params["num_lscapes"])
mutation_accuracy = []

for mutation_rate in tqdm(accuracy_params["mutation_rates"], desc="Figure 3 mutation-rate accuracy"):
    mu_results = []
    for key in rep_keys:
        start = (
            np.asarray(jr.randint(key, (accuracy_params["N"],), 0, accuracy_params["A"]), dtype=np.int32)
            if accuracy_params["random_starting_genotype"]
            else np.zeros(accuracy_params["N"], dtype=np.int32)
        )
        run = run_nk_diffusion_replicates(
            key,
            n_sites=accuracy_params["N"],
            k=accuracy_params["K"],
            num_alleles=accuracy_params["A"],
            start=start,
            popsize=accuracy_params["mutation_sweep_popsize"],
            mutation_rate=float(mutation_rate) / accuracy_params["N"],
            num_reps=accuracy_params["num_reps_per_lscape"],
            num_steps=accuracy_params["M"],
        )
        mu_results.append(run["fitness"].mean(axis=-1))
    mutation_accuracy.append(np.asarray(mu_results))

mutation_params = {
    "N": accuracy_params["N"],
    "K": accuracy_params["K"],
    "A": accuracy_params["A"],
    "num_lscapes": accuracy_params["num_lscapes"],
    "num_reps_per_lscape": accuracy_params["num_reps_per_lscape"],
    "M": accuracy_params["M"],
    "seed": accuracy_params["seed"],
    "start_policy": accuracy_params["start_policy"],
    "random_starting_genotype": accuracy_params["random_starting_genotype"],
    "popsize": accuracy_params["mutation_sweep_popsize"],
    "mutation_rates": accuracy_params["mutation_rates"],
    "rho_NK": rho_nk_accuracy_true,
}
mutation_accuracy_payload = {
    "data": np.asarray(mutation_accuracy),
    "params": mutation_params,
    "metadata": {
        "description": "Raw NK mutation-rate sensitivity sweep for Figure 3D.",
        "paper_reference": "Figure 3D",
        "output_key": "figure3_nk_mutation_accuracy",
        "filename": RAW_FILES["nk_mutation_accuracy"],
    },
}
save_pickle(mutation_accuracy_payload, RAW_DATA_DIR / RAW_FILES["nk_mutation_accuracy"])


## Processing

Load the Figure-3-specific raw payloads, compute the processed quantities needed by each panel, and save processed payloads with parameters and metadata.


In [ ]:
nk_decay_grid_payload = load_pickle(RAW_DATA_DIR / RAW_FILES["nk_decay_grid"])
popsize_accuracy_payload = load_pickle(RAW_DATA_DIR / RAW_FILES["nk_popsize_accuracy"])
mutation_accuracy_payload = load_pickle(RAW_DATA_DIR / RAW_FILES["nk_mutation_accuracy"])

nk_decay_grid = nk_decay_grid_payload["data"]
nk_decay_params = nk_decay_grid_payload["params"]
nk_pairs = np.asarray(nk_decay_params["nk_pairs"])

popsize_accuracy_raw = popsize_accuracy_payload["data"]
popsize_params = popsize_accuracy_payload["params"]
mutation_accuracy_raw = mutation_accuracy_payload["data"]
mutation_params = mutation_accuracy_payload["params"]


### Panel A Processing

Extract smooth and rugged example decay curves from the NK decay grid.


In [ ]:
smooth_rugged, fitted_lines = smooth_rugged_example(nk_decay_grid)
decay_examples_payload = {
    "data": {
        "smooth_rugged": smooth_rugged,
        "fitted_lines": fitted_lines,
        "generations": np.arange(1, len(smooth_rugged[0]) + 1),
    },
    "params": nk_decay_params,
    "metadata": {
        "description": "Processed smooth and rugged NK decay examples for Figure 3A.",
        "paper_reference": "Figure 3A",
        "output_key": "figure3_decay_examples",
        "filename": PROCESSED_FILES["decay_examples"],
    },
}
save_pickle(decay_examples_payload, PROCESSED_DATA_DIR / PROCESSED_FILES["decay_examples"])


### Panel B Processing

Fit decay rates and compare mean $\rho_{fit}$ against analytical $\rho_{NK}$.


In [ ]:
rho_NK, rho_fit = process_ruggedness_accuracy(
    nk_decay_grid,
    nk_pairs,
    steps=int(nk_decay_params["M"]),
    mutation_rate=float(nk_decay_params["mutation_rate"]),
)

order = np.argsort(rho_NK)
sorted_rho_NK = rho_NK[order]
sorted_rho_fit = rho_fit[order]
num_grid_samples = int(nk_decay_params["num_grid_samples"])
grouped_rho_NK = sorted_rho_NK.reshape(num_grid_samples, -1)
grouped_rho_fit = sorted_rho_fit.reshape(num_grid_samples, -1)

ruggedness_accuracy_processed = {
    "rho_NK": grouped_rho_NK.mean(axis=1),
    "rho_fit_mean": grouped_rho_fit.mean(axis=1),
    "rho_fit_std": grouped_rho_fit.std(axis=1),
    "rho_fit_all": rho_fit,
    "rho_NK_all": rho_NK,
}
ruggedness_accuracy_payload = {
    "data": ruggedness_accuracy_processed,
    "params": nk_decay_params,
    "metadata": {
        "description": "Processed NK ruggedness prediction accuracy for Figure 3B.",
        "paper_reference": "Figure 3B",
        "output_key": "figure3_ruggedness_accuracy",
        "filename": PROCESSED_FILES["ruggedness_accuracy"],
    },
}
save_pickle(ruggedness_accuracy_payload, PROCESSED_DATA_DIR / PROCESSED_FILES["ruggedness_accuracy"])


### Panel C-D Processing

Fit $\rho_{fit}$ for the population-size and mutation-rate robustness sweeps.


In [ ]:
popsize_processed = process_popsize_accuracy(popsize_accuracy_raw, popsize_params)
popsize_processed["rho_NK"] = np.asarray(popsize_params["rho_NK"])
popsize_accuracy_processed_payload = {
    "data": popsize_processed,
    "params": popsize_params,
    "metadata": {
        "description": "Processed NK population-size robustness data for Figure 3C.",
        "paper_reference": "Figure 3C",
        "output_key": "figure3_popsize_accuracy",
        "filename": PROCESSED_FILES["popsize_accuracy"],
    },
}
save_pickle(popsize_accuracy_processed_payload, PROCESSED_DATA_DIR / PROCESSED_FILES["popsize_accuracy"])

mutation_processed = process_mutation_accuracy(mutation_accuracy_raw, mutation_params)
mutation_processed["rho_NK"] = np.asarray(mutation_params["rho_NK"])
mutation_accuracy_processed_payload = {
    "data": mutation_processed,
    "params": mutation_params,
    "metadata": {
        "description": "Processed NK mutation-rate robustness data for Figure 3D.",
        "paper_reference": "Figure 3D",
        "output_key": "figure3_mutation_accuracy",
        "filename": PROCESSED_FILES["mutation_accuracy"],
    },
}
save_pickle(mutation_accuracy_processed_payload, PROCESSED_DATA_DIR / PROCESSED_FILES["mutation_accuracy"])


### Panel E Processing

Build the NK ruggedness metric comparison curves used in Figure 3E.


In [ ]:
metric_comparison = nk_metric_comparison_from_accuracy(rho_NK, rho_fit)
metric_names = [
    "roughness_to_slope",
    "fourier",
    "rho_fit",
    "paths_to_max",
    "closest_max",
    "rho_NK",
    "local_epistasis",
]
metric_comparison_processed = dict(zip(metric_names, metric_comparison))
metric_comparison_payload = {
    "data": metric_comparison_processed,
    "params": nk_decay_params,
    "metadata": {
        "description": "Processed NK ruggedness metric comparison for Figure 3E.",
        "paper_reference": "Figure 3E",
        "output_key": "figure3_metric_comparison",
        "filename": PROCESSED_FILES["metric_comparison"],
    },
}
save_pickle(metric_comparison_payload, PROCESSED_DATA_DIR / PROCESSED_FILES["metric_comparison"])


## Visualisation

Load the processed Figure 3 payloads and compose panels A-E into a single GridSpec figure with correct paper nomenclature.


In [ ]:
decay_examples_payload = load_pickle(PROCESSED_DATA_DIR / PROCESSED_FILES["decay_examples"])
ruggedness_accuracy_payload = load_pickle(PROCESSED_DATA_DIR / PROCESSED_FILES["ruggedness_accuracy"])
popsize_accuracy_processed_payload = load_pickle(PROCESSED_DATA_DIR / PROCESSED_FILES["popsize_accuracy"])
mutation_accuracy_processed_payload = load_pickle(PROCESSED_DATA_DIR / PROCESSED_FILES["mutation_accuracy"])
metric_comparison_payload = load_pickle(PROCESSED_DATA_DIR / PROCESSED_FILES["metric_comparison"])

decay_examples = decay_examples_payload["data"]
ruggedness_accuracy = ruggedness_accuracy_payload["data"]
popsize_accuracy_processed = popsize_accuracy_processed_payload["data"]
mutation_accuracy_processed = mutation_accuracy_processed_payload["data"]
metric_comparison = metric_comparison_payload["data"]


In [ ]:
# Figure styling.
c_decay_smooth = "tab:blue"
c_decay_rugged = "tab:orange"
c_reference = "red"
labelsize = 8
ticksize = 6
titlesize = 10
legendsize = 7
dpi = 350
plt.rcParams["font.family"] = "DejaVu Sans"

fig = plt.figure(figsize=(7.2, 6.4), dpi=dpi, constrained_layout=True)
gs = GridSpec(3, 2, figure=fig, height_ratios=[1.05, 0.82, 1.0])
axes = {
    "A": fig.add_subplot(gs[0, 0]),
    "B": fig.add_subplot(gs[0, 1]),
    "C": fig.add_subplot(gs[1, 0]),
    "D": fig.add_subplot(gs[1, 1]),
    "E": fig.add_subplot(gs[2, :]),
}

def add_panel_letter(ax, letter):
    ax.text(-0.16, 1.08, letter, transform=ax.transAxes, fontsize=12, fontweight="bold", va="top", ha="left")

# Panel A: example fitness decay curves.
ax = axes["A"]
generations = decay_examples["generations"]
smooth_rugged = decay_examples["smooth_rugged"]
fitted_lines = decay_examples["fitted_lines"]
ax.scatter(generations, smooth_rugged[1], label=r"$\rho_{NK}=0.1$", color=c_decay_smooth, s=8)
ax.plot(generations, fitted_lines[1], color=c_decay_smooth, linewidth=1)
ax.scatter(generations, smooth_rugged[0], label=r"$\rho_{NK}=0.75$", color=c_decay_rugged, s=8)
ax.plot(generations, fitted_lines[0], color=c_decay_rugged, linewidth=1)
ax.set_title("Fitness decay curves", fontsize=titlesize)
ax.set_xlabel(r"Generations $M$", fontsize=labelsize)
ax.set_ylabel(r"Fitness $F_\mu$", fontsize=labelsize)
ax.legend(fontsize=legendsize, frameon=False)
add_panel_letter(ax, "A")

# Panel B: ruggedness prediction accuracy.
ax = axes["B"]
rho_NK = ruggedness_accuracy["rho_NK"]
rho_fit_mean = ruggedness_accuracy["rho_fit_mean"]
rho_fit_std = ruggedness_accuracy["rho_fit_std"]
ax.plot(rho_NK, rho_fit_mean, "o-", label=r"Mean $\rho_{fit}$", markersize=3)
ax.fill_between(rho_NK, rho_fit_mean - rho_fit_std, rho_fit_mean + rho_fit_std, alpha=0.25, label=r"$\pm$1 s.d.")
ax.plot(rho_NK, rho_NK, color=c_reference, alpha=0.45, linestyle="--", label=r"$\rho_{NK}$")
ax.set_title(r"$\rho_{fit}$ prediction accuracy", fontsize=titlesize)
ax.set_xlabel(r"$\rho_{NK}$", fontsize=labelsize)
ax.set_ylabel(r"Mean $\rho_{fit}$", fontsize=labelsize)
ax.legend(fontsize=legendsize, frameon=False)
ax.grid(True, alpha=0.3)
add_panel_letter(ax, "B")

# Panel C: population-size robustness.
ax = axes["C"]
pop_rates = popsize_accuracy_processed["rates"]
pop_sizes = popsize_accuracy_processed["pop_sizes"]
pop_means = pop_rates.mean(axis=1)
pop_stds = pop_rates.std(axis=1)
ax.plot(pop_sizes, pop_means, "o-", label=r"Mean $\rho_{fit}$", markersize=3)
ax.fill_between(pop_sizes, pop_means - pop_stds, pop_means + pop_stds, alpha=0.25, label=r"$\pm$1 s.d.")
ax.axhline(float(popsize_accuracy_processed["rho_NK"]), color=c_reference, alpha=0.45, linestyle="--", label=r"$\rho_{NK}$")
ax.set_xscale("log")
ax.set_title("Accuracy over population size", fontsize=titlesize)
ax.set_xlabel("Population size", fontsize=labelsize)
ax.set_ylabel(r"Mean $\rho_{fit}$", fontsize=labelsize)
ax.legend(fontsize=legendsize, frameon=False)
ax.grid(True, alpha=0.3)
add_panel_letter(ax, "C")

# Panel D: mutation-rate robustness.
ax = axes["D"]
mut_rates = mutation_accuracy_processed["rates"]
mutation_rates = mutation_accuracy_processed["mutation_rates"]
mut_means = mut_rates.mean(axis=1)
mut_stds = mut_rates.std(axis=1)
ax.plot(mutation_rates, mut_means, "o-", label=r"Mean $\rho_{fit}$", markersize=3)
ax.fill_between(mutation_rates, mut_means - mut_stds, mut_means + mut_stds, alpha=0.25, label=r"$\pm$1 s.d.")
ax.axhline(float(mutation_accuracy_processed["rho_NK"]), color=c_reference, alpha=0.45, linestyle="--", label=r"$\rho_{NK}$")
ax.set_title("Accuracy over mutation rate", fontsize=titlesize)
ax.set_xlabel(r"Mutations per cell per generation $\theta$", fontsize=labelsize)
ax.set_ylabel(r"Mean $\rho_{fit}$", fontsize=labelsize)
ax.legend(fontsize=legendsize, frameon=False)
ax.grid(True, alpha=0.3)
add_panel_letter(ax, "D")

# Panel E: ruggedness metric comparison.
ax = axes["E"]
rho_axis = metric_comparison["rho_NK"]
rho_curve = metric_comparison["rho_fit"]
ax.plot(rho_axis, rho_curve / rho_curve.max(), label=r"$\rho_{fit}$", linewidth=1.5)
ax.plot(rho_axis, 1 - metric_comparison["fourier"] / metric_comparison["fourier"].max(), label=r"1 - Landscape $R^2$", alpha=0.75, linestyle="--")
ax.plot(rho_axis, metric_comparison["roughness_to_slope"] / metric_comparison["roughness_to_slope"].max(), label="Roughness-to-slope ratio", alpha=0.75, linestyle="--")
ax.plot(rho_axis, 1 - metric_comparison["paths_to_max"] / metric_comparison["paths_to_max"].max(), label="1 - Paths to maximum", alpha=0.75, linestyle="--")
ax.plot(rho_axis, metric_comparison["closest_max"] / metric_comparison["closest_max"].max(), label="Distance to closest local maximum", alpha=0.75, linestyle="--")
ax.plot(rho_axis, metric_comparison["local_epistasis"] / metric_comparison["local_epistasis"].max(), label="Local epistasis", alpha=0.75, linestyle="--")
ax.set_title("Ruggedness metric comparison", fontsize=titlesize)
ax.set_xlabel(r"$\rho_{NK}$", fontsize=labelsize)
ax.set_ylabel("Normalised ruggedness measurements", fontsize=labelsize)
ax.legend(loc="lower right", fontsize=legendsize, frameon=False, ncol=2)
ax.grid(True, alpha=0.3)
add_panel_letter(ax, "E")

for ax in axes.values():
    ax.tick_params(axis="both", which="major", labelsize=ticksize)

for figure_type in ("pdf", "png", "eps"):
    fig.savefig(FIGURES_DIR / figure_type / f"figure_3.{figure_type}", dpi=dpi)
plt.show()
